# SBA 7(a) Loan Risk — Feasibility and Source-Data Assessment

**Stage:** Exploratory feasibility review, before PostgreSQL modeling and final feature selection.

This notebook inspects the original SBA FOIA loan-level CSV, reconciles available columns with the supplied data dictionary, reviews statuses and date fields, and explores potential five-year charge-off outcomes. It also examines duplicate source rows and data-quality issues.

**Run location:** Open Jupyter with its working directory set to the repository’s `notebooks/` folder. Source files are expected in `../data/raw/` and are not committed to GitHub.

**Important:** Exploratory counts and comparisons here are not independent underwriting estimates. The SQL analytical views and subsequent modeling notebooks define the authoritative final modeling cohort, target, and model choice. Source duplicates were investigated, not automatically dropped. Never use borrower identifying columns as model features.


In [ ]:

import pandas as pd
from pathlib import Path

# Main project directory
project_dir = Path.cwd().parent

# Original SBA CSV
file_path = project_dir / "data" / "raw" / "FOIA_7a_FY2010_FY2019_asof_260630.csv"

# Verify the expected local input exists
if not file_path.is_file():
    raise FileNotFoundError(f"Missing SBA CSV: {file_path}. Run from notebooks/ and obtain the public SBA file.")

# Read the data
df = pd.read_csv(file_path, low_memory=False)

# Verify dimensions
print("Dataset shape:", df.shape)


Dataset shape: (545751, 42)


In [ ]:

# Display information about the DataFrame
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545751 entries, 0 to 545750
Data columns (total 42 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   AsOfDate                    545751 non-null  object 
 1   Program                     545751 non-null  object 
 2   LocationID                  545750 non-null  float64
 3   BorrName                    545721 non-null  object 
 4   BorrStreet                  545751 non-null  object 
 5   BorrCity                    545751 non-null  object 
 6   BorrState                   545751 non-null  object 
 7   BorrZip                     545751 non-null  int64  
 8   BankName                    545750 non-null  object 
 9   BankFDICNumber              506442 non-null  float64
 10  BankNCUANumber              17691 non-null   float64
 11  BankStreet                  545750 non-null  object 
 12  BankCity                    545750 non-null  object 
 13  BankState     

In [ ]:

# Display all column names
print(df.columns.tolist())

# Display a small sample without personal identifiers
sample_columns = [
    "ApprovalDate",
    "ApprovalFY",
    "GrossApproval",
    "SBAGuaranteedApproval",
    "LoanStatus",
    "TermInMonths",
    "InitialInterestRate",
    "ProcessingMethod",
    "BusinessAge",
    "CollateralInd"
]

display(df[sample_columns].head())


['AsOfDate', 'Program', 'LocationID', 'BorrName', 'BorrStreet', 'BorrCity', 'BorrState', 'BorrZip', 'BankName', 'BankFDICNumber', 'BankNCUANumber', 'BankStreet', 'BankCity', 'BankState', 'BankZip', 'GrossApproval', 'SBAGuaranteedApproval', 'ApprovalDate', 'ApprovalFY', 'FirstDisbursementDate', 'ProcessingMethod', 'InitialInterestRate', 'FixedorVariableInterestInd', 'TermInMonths', 'NaicsCode', 'NaicsDescription', 'FranchiseCode', 'FranchiseName', 'ProjectCounty', 'ProjectState', 'SBADistrictOffice', 'CongressionalDistrict', 'BusinessType', 'BusinessAge', 'LoanStatus', 'PaidInFullDate', 'ChargeOffDate', 'GrossChargeOffAmount', 'RevolverStatus', 'JobsSupported', 'CollateralInd', 'SoldSecMrktInd']


,ApprovalDate,ApprovalFY,GrossApproval,SBAGuaranteedApproval,LoanStatus,TermInMonths,InitialInterestRate,ProcessingMethod,BusinessAge,CollateralInd
0,2009-10-01,2010,288000.0,259200.0,P I F,120.0,6.00,Preferred Lenders Program,Unanswered,Y
1,2009-10-01,2010,1200000.0,1080000.0,P I F,300.0,4.75,Preferred Lenders Program,"Existing, 5 or more years",N
2,2009-10-01,2010,120000.0,108000.0,P I F,90.0,5.25,Preferred Lenders Program,Less than 4 years old but at least 3,N
3,2009-10-01,2010,150000.0,75000.0,P I F,60.0,5.25,SBA Express Program,"Startup, Loan Funds will Open Business",Y
4,2009-10-01,2010,790000.0,711000.0,CANCLD,240.0,6.00,Preferred Lenders Program,"New, Less than 1 Year old",Y


In [ ]:

# Count records by loan status
status_counts = df["LoanStatus"].value_counts(dropna=False)

print(status_counts)

# Verify total records
print("\nTotal records:", status_counts.sum())


LoanStatus
P I F     394721
CANCLD     66744
EXEMPT     50089
CHGOFF     34153
COMMIT        44
Name: count, dtype: int64

Total records: 545751


In [ ]:

# Path to the official SBA data dictionary
dictionary_path = (
    project_dir / "data" / "raw" /
    "7a_504_foia_data_dictionary.xlsx"
)

# Open the Excel workbook
dictionary = pd.ExcelFile(dictionary_path)

# Display worksheet names
print("Dictionary worksheets:")
print(dictionary.sheet_names)


Dictionary worksheets:
['7(a) Data Dictionary', '504 Data Dictionary']


In [ ]:

# Load only the 7(a) dictionary
dict_df = pd.read_excel(
    dictionary_path,
    sheet_name="7(a) Data Dictionary"
)

# Check the dictionary structure
print("Dictionary shape:", dict_df.shape)

# Display the first 10 rows
display(dict_df.head(10))


Dictionary shape: (42, 2)


,Field Name,Definition
0,AsOfDate,Date when the data was recorded
1,Program,Indicator of whether loan was approved under S...
2,LocationID,SBA's unique lender ID code
3,BorrName,Borrower name
4,BorrStreet,Borrower street address
5,BorrCity,Borrower city
6,BorrState,Borrower state
7,BorrZip,Borrower zip code
8,BankName,Name of the bank that the loan is currently as...
9,BankFDICNumber,The Federal Depository Insurance Corporation c...


In [ ]:

# Extract field names from both sources
csv_columns = set(df.columns)
dictionary_columns = set(dict_df["Field Name"].dropna())

# Compare the fields
print("CSV column count:", len(csv_columns))
print("Dictionary field count:", len(dictionary_columns))

print("\nCSV fields missing from dictionary:")
print(csv_columns - dictionary_columns)

print("\nDictionary fields missing from CSV:")
print(dictionary_columns - csv_columns)


CSV column count: 42
Dictionary field count: 42

CSV fields missing from dictionary:
set()

Dictionary fields missing from CSV:
set()


In [ ]:

# Select fields relevant to loan outcomes
important_fields = [
    "LoanStatus",
    "ApprovalDate",
    "ApprovalFY",
    "ChargeOffDate",
    "PaidInFullDate",
    "FirstDisbursementDate"
]

# Filter the official dictionary
selected_definitions = dict_df[
    dict_df["Field Name"].isin(important_fields)
]

# Display complete definitions without truncation
for _, row in selected_definitions.iterrows():
    print("Field:", row["Field Name"])
    print("Definition:", row["Definition"])
    print("-" * 60)


Field: ApprovalDate
Definition: Date the loan was approved
------------------------------------------------------------
Field: ApprovalFY
Definition: Fiscal year the loan was approved
------------------------------------------------------------
Field: FirstDisbursementDate
Definition: Date of first loan disbursement (if available)
------------------------------------------------------------
Field: LoanStatus
Definition: Current status of loan:  
• CANCLD = Cancelled
• CHGOFF = Charged Off
• COMMIT = Undisbursed
• EXEMPT = The status of loans that have been disbursed but have not been cancelled, paid in full, or charged off are exempt from disclosure under FOIA Exemption 4
• PIF = Paid In Full
------------------------------------------------------------
Field: PaidInFullDate
Definition: Date loan was paid in full (if applicable)
------------------------------------------------------------
Field: ChargeOffDate
Definition: Date SBA charged off loan (if applicable)
------------------------

In [ ]:

# Verify the reporting snapshot
print("Reporting dates:")
print(df["AsOfDate"].value_counts(dropna=False))

# Inspect approval date range
print("\nEarliest approval:", df["ApprovalDate"].min())
print("Latest approval:", df["ApprovalDate"].max())

# Verify approval fiscal years
print("\nApproval fiscal years:")
print(df["ApprovalFY"].value_counts().sort_index())


Reporting dates:
AsOfDate
2026-06-30    545751
Name: count, dtype: int64

Earliest approval: 2009-10-01
Latest approval: 2019-09-30

Approval fiscal years:
ApprovalFY
2010    47000
2011    53710
2012    44376
2013    46395
2014    52044
2015    63461
2016    64074
2017    62430
2018    60354
2019    51907
Name: count, dtype: int64


In [ ]:

# Identify charged-off loans
charged_off = df["LoanStatus"] == "CHGOFF"

# Identify missing charge-off dates
missing_date = df["ChargeOffDate"].isna()

# Count records satisfying both conditions
missing_chargeoff_dates = df[
    charged_off & missing_date
]

print(
    "Charged-off loans missing a charge-off date:",
    len(missing_chargeoff_dates)
)

# Display selected fields without borrower identifiers
display(
    missing_chargeoff_dates[
        [
            "ApprovalFY",
            "ApprovalDate",
            "LoanStatus",
            "ChargeOffDate",
            "GrossChargeOffAmount"
        ]
    ].head(10)
)



Charged-off loans missing a charge-off date: 10


,ApprovalFY,ApprovalDate,LoanStatus,ChargeOffDate,GrossChargeOffAmount
46331,2010,2010-09-23,CHGOFF,NaN,0.0
231862,2014,2014-07-23,CHGOFF,NaN,0.0
255670,2015,2014-12-18,CHGOFF,NaN,0.0
326429,2016,2016-02-02,CHGOFF,NaN,0.0
334828,2016,2016-03-22,CHGOFF,NaN,0.0
343710,2016,2016-05-06,CHGOFF,NaN,0.0
354930,2016,2016-07-07,CHGOFF,NaN,0.0
420007,2017,2017-07-23,CHGOFF,NaN,0.0
478729,2018,2018-07-05,CHGOFF,NaN,0.0
530544,2019,2019-06-19,CHGOFF,NaN,0.0


In [ ]:

# Convert dates for validation without modifying the raw columns
approval_dates = pd.to_datetime(
    df["ApprovalDate"], errors="coerce"
)

chargeoff_dates = pd.to_datetime(
    df["ChargeOffDate"], errors="coerce"
)

snapshot_dates = pd.to_datetime(
    df["AsOfDate"], errors="coerce"
)

# Identify charged-off loans with valid recorded dates
valid_chargeoffs = (
    (df["LoanStatus"] == "CHGOFF") &
    chargeoff_dates.notna()
)

# Check chronology
before_approval = valid_chargeoffs & (
    chargeoff_dates < approval_dates
)

after_snapshot = valid_chargeoffs & (
    chargeoff_dates > snapshot_dates
)

print("Charged-off loans with usable dates:",
      valid_chargeoffs.sum())

print("Charge-offs before approval:",
      before_approval.sum())

print("Charge-offs after reporting snapshot:",
      after_snapshot.sum())

print("Unparseable approval dates:",
      approval_dates.isna().sum())


Charged-off loans with usable dates: 34143
Charge-offs before approval: 0
Charge-offs after reporting snapshot: 9
Unparseable approval dates: 0


In [ ]:

# Inspect charge-offs dated after the reporting snapshot
future_chargeoffs = df.loc[
    after_snapshot,
    [
        "ApprovalFY",
        "ApprovalDate",
        "ChargeOffDate",
        "LoanStatus",
        "GrossApproval",
        "GrossChargeOffAmount"
    ]
]

display(future_chargeoffs)


,ApprovalFY,ApprovalDate,ChargeOffDate,LoanStatus,GrossApproval,GrossChargeOffAmount
303365,2015,2015-09-15,2026-07-01,CHGOFF,75000.0,21781.08
329511,2016,2016-02-22,2026-07-01,CHGOFF,150000.0,7577.29
345306,2016,2016-05-16,2026-07-01,CHGOFF,150000.0,87115.00
373975,2017,2016-10-21,2026-07-01,CHGOFF,100000.0,60381.72
448979,2018,2017-12-28,2026-07-01,CHGOFF,456500.0,385921.43
491472,2018,2018-09-21,2026-07-01,CHGOFF,202500.0,81616.09
512222,2019,2019-02-13,2026-07-01,CHGOFF,10000.0,8502.45
529694,2019,2019-06-13,2026-07-01,CHGOFF,215000.0,27391.35
533153,2019,2019-07-08,2026-07-01,CHGOFF,100000.0,129711.56


In [ ]:

# Select charge-offs with dates within the reporting window
valid_events = (
    (df["LoanStatus"] == "CHGOFF") &
    chargeoff_dates.notna() &
    (chargeoff_dates >= approval_dates) &
    (chargeoff_dates <= snapshot_dates)
)

# Calculate elapsed time from approval to charge-off
months_to_chargeoff = (
    (chargeoff_dates[valid_events] -
     approval_dates[valid_events]).dt.days / 30.44
)

# Summary statistics
print("Valid charge-off events:", valid_events.sum())

print("\nMonths from approval to charge-off:")
print(
    months_to_chargeoff.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90]
    ).round(2)
)

# Count events within approximately 60 months
within_60_months = (months_to_chargeoff <= 60).sum()

print("\nCharge-offs within approximately 60 months:",
      within_60_months)


Valid charge-off events: 34134

Months from approval to charge-off:
count    34134.00
mean        59.20
std         28.71
min          5.29
25%         37.12
50%         55.85
75%         77.46
90%         96.48
max        197.73
dtype: float64

Charge-offs within approximately 60 months: 18821


In [ ]:

# Calculate the exact five-year anniversary
five_year_cutoff = approval_dates + pd.DateOffset(years=5)

# Identify valid charge-offs within five years
chargeoff_60m = (
    valid_events &
    (chargeoff_dates <= five_year_cutoff)
)

print("Valid charge-offs within five years:",
      chargeoff_60m.sum())

print(
    "Percentage of valid charge-offs within five years:",
    round(chargeoff_60m.sum() / valid_events.sum() * 100, 2),
    "%"
)

print(
    "Difference from approximate 60-month method:",
    chargeoff_60m.sum() - within_60_months
)


Valid charge-offs within five years: 18826
Percentage of valid charge-offs within five years: 55.15 %
Difference from approximate 60-month method: 5


In [ ]:

# Identify the disbursed-portfolio status population
eligible_status = df["LoanStatus"].isin(
    ["P I F", "CHGOFF", "EXEMPT"]
)

# Identify charged-off records with missing event dates
unknown_chargeoff_timing = (
    (df["LoanStatus"] == "CHGOFF") &
    chargeoff_dates.isna()
)

# Define population with known five-year outcomes
eligible_population = (
    eligible_status &
    ~unknown_chargeoff_timing
)

print("Total loan records:", len(df))

print(
    "Records with eligible statuses:",
    eligible_status.sum()
)

print(
    "Charged-off records with unknown timing:",
    unknown_chargeoff_timing.sum()
)

print(
    "Initial eligible population:",
    eligible_population.sum()
)

print(
    "\nFive-year charge-off events:",
    (chargeoff_60m & eligible_population).sum()
)


Total loan records: 545751
Records with eligible statuses: 478963
Charged-off records with unknown timing: 10
Initial eligible population: 478953

Five-year charge-off events: 18826


In [ ]:

# Calculate five-year charge-off incidence

total_eligible = eligible_population.sum()

five_year_events = (
    chargeoff_60m & eligible_population
).sum()

chargeoff_rate_5yr = (
    five_year_events / total_eligible
) * 100

print("Eligible loans:", total_eligible)
print("Five-year charge-offs:", five_year_events)
print(
    f"Five-year charge-off incidence: {chargeoff_rate_5yr:.2f}%"
)


Eligible loans: 478953
Five-year charge-offs: 18826
Five-year charge-off incidence: 3.93%


In [ ]:

# Create a temporary cohort analysis table
vintage_df = df.loc[
    eligible_population,
    ["ApprovalFY"]
].copy()

# Add the five-year charge-off indicator
vintage_df["ChargeOff_5yr"] = (
    chargeoff_60m.loc[eligible_population].astype(int)
)

# Aggregate by approval fiscal year
vintage_summary = (
    vintage_df
    .groupby("ApprovalFY")
    .agg(
        TotalLoans=("ChargeOff_5yr", "count"),
        ChargeOffs_5yr=("ChargeOff_5yr", "sum")
    )
    .reset_index()
)

# Calculate the incidence percentage
vintage_summary["ChargeOffRate_5yr"] = (
    vintage_summary["ChargeOffs_5yr"] /
    vintage_summary["TotalLoans"] * 100
)

display(vintage_summary.round(2))


,ApprovalFY,TotalLoans,ChargeOffs_5yr,ChargeOffRate_5yr
0,2010,39912,1955,4.90
1,2011,45628,1667,3.65
2,2012,38889,1276,3.28
3,2013,40416,1242,3.07
4,2014,45961,1617,3.52
5,2015,55420,2137,3.86
6,2016,56785,2198,3.87
7,2017,56078,2419,4.31
8,2018,54194,2529,4.67
9,2019,45670,1786,3.91


In [ ]:

# Compare loan status with first-disbursement date availability
disbursement_check = pd.crosstab(
    df["LoanStatus"],
    df["FirstDisbursementDate"].notna(),
    margins=True
)

# Make the column labels easier to interpret
disbursement_check = disbursement_check.rename(
    columns={
        False: "Date Missing",
        True: "Date Present"
    }
)

display(disbursement_check)


FirstDisbursementDate,Date Missing,Date Present,All
LoanStatus,,,
CANCLD,66744,0,66744
CHGOFF,10,34143,34153
COMMIT,44,0,44
EXEMPT,1,50088,50089
P I F,252,394469,394721
All,67051,478700,545751


In [ ]:

# Charged-off loans with missing first-disbursement dates
missing_disbursement_chgoff = (
    (df["LoanStatus"] == "CHGOFF") &
    df["FirstDisbursementDate"].isna()
)

# Check overlap with missing charge-off dates
both_dates_missing = (
    missing_disbursement_chgoff &
    df["ChargeOffDate"].isna()
)

print(
    "CHGOFF records missing disbursement date:",
    missing_disbursement_chgoff.sum()
)

print(
    "CHGOFF records missing both dates:",
    both_dates_missing.sum()
)

print(
    "CHGOFF records missing disbursement date only:",
    (missing_disbursement_chgoff & df["ChargeOffDate"].notna()).sum()
)


CHGOFF records missing disbursement date: 10
CHGOFF records missing both dates: 0
CHGOFF records missing disbursement date only: 10


In [ ]:

# Identify exact duplicate rows
duplicate_mask = df.duplicated(keep="first")

# Count duplicates
print("Exact duplicate rows:", duplicate_mask.sum())

# Examine duplicate distribution by loan status
duplicate_status = (
    df.loc[duplicate_mask, "LoanStatus"]
    .value_counts()
)

print("\nDuplicate rows by loan status:")
print(duplicate_status)


Exact duplicate rows: 1136

Duplicate rows by loan status:
LoanStatus
CANCLD    824
P I F     291
EXEMPT     21
Name: count, dtype: int64


In [ ]:

# Scenario A: Keep all eligible records
loans_with_duplicates = eligible_population.sum()

events_with_duplicates = (
    eligible_population & chargeoff_60m
).sum()

rate_with_duplicates = (
    events_with_duplicates / loans_with_duplicates * 100
)

# Scenario B: Exclude repeated occurrences
eligible_without_duplicates = (
    eligible_population & ~duplicate_mask
)

loans_without_duplicates = (
    eligible_without_duplicates.sum()
)

events_without_duplicates = (
    eligible_without_duplicates & chargeoff_60m
).sum()

rate_without_duplicates = (
    events_without_duplicates /
    loans_without_duplicates * 100
)

# Compare the scenarios
print("Scenario A — All eligible records")
print("Loans:", loans_with_duplicates)
print("5-year charge-offs:", events_with_duplicates)
print(f"Rate: {rate_with_duplicates:.4f}%")

print("\nScenario B — Excluding duplicate occurrences")
print("Loans:", loans_without_duplicates)
print("5-year charge-offs:", events_without_duplicates)
print(f"Rate: {rate_without_duplicates:.4f}%")

print(
    "\nDifference in percentage points:",
    round(rate_without_duplicates - rate_with_duplicates, 4)
)


Scenario A — All eligible records
Loans: 478953
5-year charge-offs: 18826
Rate: 3.9307%

Scenario B — Excluding duplicate occurrences
Loans: 478641
5-year charge-offs: 18826
Rate: 3.9332%

Difference in percentage points: 0.0026


In [ ]:

# Identify potentially problematic financial values
invalid_term = df["TermInMonths"] <= 0

unusual_long_term = df["TermInMonths"] > 360

nonpositive_interest = df["InitialInterestRate"] <= 0

# Count each condition
print("Loans with term <= 0:", invalid_term.sum())

print("Loans with term > 360 months:",
      unusual_long_term.sum())

print("Loans with interest rate <= 0:",
      nonpositive_interest.sum())

# Examine how these issues relate to loan status
quality_flags = pd.DataFrame({
    "LoanStatus": df["LoanStatus"],
    "InvalidTerm": invalid_term,
    "LongTerm": unusual_long_term,
    "NonpositiveInterest": nonpositive_interest
})

quality_summary = (
    quality_flags
    .groupby("LoanStatus")[
        ["InvalidTerm", "LongTerm", "NonpositiveInterest"]
    ]
    .sum()
)

display(quality_summary)


Loans with term <= 0: 229
Loans with term > 360 months: 13
Loans with interest rate <= 0: 8


,InvalidTerm,LongTerm,NonpositiveInterest
LoanStatus,,,
CANCLD,1,3,1
CHGOFF,165,0,0
COMMIT,0,0,0
EXEMPT,6,7,2
P I F,57,3,5


In [ ]:

# Inspect loans with nonpositive terms
term_issues = df.loc[
    df["TermInMonths"] <= 0,
    [
        "TermInMonths",
        "LoanStatus",
        "ProcessingMethod",
        "ApprovalFY"
    ]
].copy()

# Examine actual term values
print("Nonpositive term values:")
print(term_issues["TermInMonths"].value_counts(dropna=False))

# Examine processing methods
print("\nProcessing methods:")
print(term_issues["ProcessingMethod"].value_counts())

# Examine approval fiscal years
print("\nApproval years:")
print(term_issues["ApprovalFY"].value_counts().sort_index())


Nonpositive term values:
TermInMonths
0.0    229
Name: count, dtype: int64

Processing methods:
ProcessingMethod
SBA Express Program                    162
Preferred Lenders Program               29
Patriot Express Loans                   10
Community Express                        6
Small Loan Advantage Initiative          5
7a General                               4
Community Advantage Initiative           3
7a with EWCP                             2
Gulf Opportunity Pilot Loan Program      2
Contract CAPLine                         2
Rural Loan Initiative                    2
Working Capital CAPLine                  1
Export Express                           1
Name: count, dtype: int64

Approval years:
ApprovalFY
2010    20
2011    36
2012    28
2013    16
2014    19
2015    33
2016    26
2017    22
2018    23
2019     6
Name: count, dtype: int64
